# Chapter 6: Evaluating Multi-Turn Conversations
Session-level judge, grouping by length, a user simulator, isolating a failure, and perturbation tests.

In [ ]:
from common import ask, chat, load_data
import json

data = load_data()["ch6"]

def transcript_text(turns):
    return "\n".join(f"{role.upper()}: {text}" for role, text in turns)

## 1. Session-level judge
One Pass/Fail for the **whole conversation**: did the user get what they wanted? (Start here, not with turn-level scoring.)

In [ ]:
def session_judge(goal, turns):
    prompt = f"""User goal: {goal}
Success means: {data["session_success_definition"]}

Conversation:
{transcript_text(turns)}

Reply as JSON: {{"reasoning": "one short sentence", "answer": "Pass" or "Fail"}}"""
    return ask(prompt, as_json=True)

agree = 0
for conv in data["conversations"]:
    out = session_judge(conv["goal"], conv["transcript"])
    ok = out["answer"] == conv["human_label"]
    agree += ok
    print(f"{conv['id']}: judge={out['answer']:4} human={conv['human_label']:4} {'OK' if ok else 'MISMATCH'} | {out['reasoning']}")
print(f"\nAgreement with human labels: {agree}/{len(data['conversations'])}")

## 2. Group results by conversation length
Many agents work in short chats and break in long ones.

In [ ]:
import pandas as pd

df = pd.DataFrame(data["session_results"])

def bucket(n):
    if n <= 4: return "short (2-4)"
    if n <= 9: return "medium (5-9)"
    return "long (10+)"

df["length"] = df["num_turns"].apply(bucket)
print(df.groupby("length")["success"].agg(["count", "mean"]).rename(columns={"mean": "success_rate"}))

## 3. User simulator
A second LLM **plays the user** (persona + goal). The first LLM is the assistant under test.

In [ ]:
houses = json.dumps(data["houses"])
assistant_system = data["assistant_system"].replace("{houses}", houses)

def simulate(persona, max_turns=6):
    history = []                                        # real chat from the assistant's point of view
    for turn in range(max_turns):
        # the simulator sees the chat with roles flipped (it is the "assistant" of this view)
        view = "\n".join(f"{'YOU' if m['role'] == 'user' else 'ASSISTANT'}: {m['content']}" for m in history)
        user_msg = ask(f"Conversation so far:\n{view or '(empty)'}\n\nWrite your next message.",
                       system=persona, temperature=0.7).strip()
        if "[DONE]" in user_msg:
            break
        history.append({"role": "user", "content": user_msg})
        history.append({"role": "assistant", "content": chat(history, system=assistant_system)})
    return history

sim_history = simulate(data["simulator_persona"])
for m in sim_history:
    print(f"{m['role'].upper():9}: {m['content']}\n")

In [ ]:
# Judge the simulated conversation with the session judge
turns = [(m["role"], m["content"]) for m in sim_history]
print(session_judge("Find a house with at least 3 bedrooms under $500k", turns))

## 4. Isolate the failure: multi-turn or single-turn?
Put the whole conversation into **one** prompt. If it still fails, the problem is *not* about conversation memory.

In [ ]:
def check_answer(text, rule):
    """Code-based check: right houses mentioned, wrong houses not mentioned."""
    include_ok = any(k in text for k in rule["must_include_any"]) if rule["must_include_any"] else True
    exclude_ok = not any(k in text for k in rule["must_exclude"])
    return include_ok and exclude_ok

rule = data["base_check"]
user_turns = data["base_user_turns"]

# A) real multi-turn chat
history = []
for u in user_turns:
    history.append({"role": "user", "content": u})
    history.append({"role": "assistant", "content": chat(history, system=assistant_system)})
multi_answer = history[-1]["content"]

# B) everything in ONE single prompt
single_prompt = " ".join(user_turns)
single_answer = ask(single_prompt, system=assistant_system)

print("Multi-turn passes :", check_answer(multi_answer, rule))
print("Single-turn passes:", check_answer(single_answer, rule))
print("\nMulti-turn answer:\n", multi_answer)

## 5. Perturbation tests + brittleness
Take the **passing** conversation, change one thing, and see if the agent still behaves.

In [ ]:
results = []
for p in data["perturbations"]:
    h = [dict(m) for m in history]                         # copy of the passing conversation
    h.append({"role": "user", "content": p["message"]})
    answer = chat(h, system=assistant_system)
    ok = check_answer(answer, p)
    results.append(ok)
    print(f"{p['name']:20} -> {'survived' if ok else 'BROKE'}\n   {answer[:150]}\n")

broke = results.count(False)
print(f"Broke on {broke} of {len(results)} perturbations")
if False in results:
    print("First break at perturbation #", results.index(False) + 1)